In [1]:
import pandas as pd
import scanpy as sc 
import scanpy.external as sce
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import numpy as np
import cosg

In [2]:
def read_10x_manually(dir_datset):
    adata = sc.read_mtx(filename= dir_datset + '/seu_final/matrix.mtx')
    adata_bc=pd.read_csv(filepath_or_buffer= dir_datset + '/seu_final/barcodes.tsv', header=None)
    adata_features=pd.read_csv(filepath_or_buffer= dir_datset + '/seu_final/features.tsv',header=None)
    adata_obs=pd.read_csv(filepath_or_buffer= dir_datset + '/seu_final/metadata.tsv', sep='\t')
    adata= adata.T
    adata.obs=adata_obs
    adata.obs.index=adata_bc[0].tolist()
    adata.var['gene_name']= adata_features[0].tolist()
    adata.var.index= adata.var['gene_name']
    print(adata.obs['cohort'].unique())
    return adata

In [3]:
dataset_names = ['SCC_Yost', 'NSCLC_Liu','SKCM_Becker', 'SKCM_Plozniak', 'BCC_Yost', 
                'BRCA_Bassez1', 'BRCA_Bassez2', 'TNBC_Zhang', 'TNBC_Shiao', 'HNSC_Franken', 'HNSC_vanderLeun', 'HNSC_Luoma', 
                'NSCLC_Yan', 'NSCLC_Liu',
                'CRC_Li', 'CRC_Chen', 'PCa_Hawley','HCC_Guo','HCC_Ma','RCC_Bi']
datasets = [read_10x_manually(dir_datset='/bigdata/zlin/PanCancer_ICI/data/' + name) for name in dataset_names]

['SCC_Yost']
['NSCLC_Liu']
['SKCM_this study']


/tmp/ipykernel_3434969/3559320569.py:5: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  adata_obs=pd.read_csv(filepath_or_buffer= dir_datset + '/seu_final/metadata.tsv', sep='\t')


['SKCM_Plozniak']


/tmp/ipykernel_3434969/3559320569.py:5: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  adata_obs=pd.read_csv(filepath_or_buffer= dir_datset + '/seu_final/metadata.tsv', sep='\t')


['BCC_Yost']
['BRCA_Bassez1']
['BRCA_Bassez2']
['TNBC_Zhang']
['TNBC_Shiao']
['HNSC_Franken']
['HNSC_vanderLeun']
['HNSC_Luoma']
['NSCLC_Yan']
['NSCLC_Liu']
['CRC_Li']
['CRC_Chen']
['PCa_Hawley']
['HCC_Guo']
['HCC_Ma']
['RCC_Bi']


In [ ]:
adata = ad.concat(datasets, join='outer')

In [ ]:
adata.obs_names_make_unique()
adata.obs['cohort'] = adata.obs['cohort'].astype(str)
adata.obs['celltype_r2'] = adata.obs['celltype_r2'].astype(str)

In [9]:
adata.obs.loc[(adata.obs['celltype_main']=='Malignant(CNA+)') & (adata.obs['celltype_major']=='Epithelial cells'), 'celltype_r2'] = 'Epithelial(CNA+)'

In [10]:
adata.obs.loc[adata.obs['cohort'].isin(['SCC_Yost','BCC_Yost']), 'cohort'] = 'BCCSCC_Yost'

adata.obs.loc[(adata.obs['celltype_main']=='Malignant(CNA+)') & (adata.obs['celltype_major']=='Melanocytes'), 'celltype_r2'] = 'Melanocytes(CNA+)'

In [11]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.raw = adata

In [6]:
n_gene=100
groupby='celltype_r2'
cosg.cosg(
   adata,
   key_added='cosg',
   batch_key='cohort',
   batch_cell_number_threshold=50,
   mu=100,
   expressed_pct=0.1,
   remove_lowly_expressed=True,
   n_genes_user=100,
   groupby=groupby
)

In [ ]:
# maximum number of columns
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [12]:
pd.DataFrame(adata.uns['cosg']['names']).head(n=30)

,ACB_CCR7,ACB_EGR1,ACB_NR4A2,B-AtM,B-HSP,B-ISG,B-memory,B-naive,CAF-ap,CAF-desmo,CAF-prog,CAF_SFRP2,CD4_T-ISG,CD4_T-naive,CD4_Tcm,CD4_Tctl,CD4_Tfh,CD4_Th17,CD4_Treg,CD4_Tstr,CD8_NK-like,CD8_T-ISG,CD8_T-naive,CD8_Tem,CD8_Tem-early,CD8_Temra,CD8_Tex_CXCL13,CD8_Tex_GZMK,CD8_Tm,CD8_Tpex,CD8_Trm,CD8_Tstr,DC_LC-like,Endo-artery,Endo-capillary,Endo-lymphatic,Endo-tip,Endo-vein,Epithelial(CNA-),GCB-DZ_SUGCT,GCB-LZ_LMO2,GCB-cycling,GCB-pre,MAIT,Macro-ISG,Macro_C1QC,Macro_FN1,Macro_IL1B,Macro_INHBA,Macro_LYVE1,Macro_SPP1,Macro_TNF,Macro_TREM2,Malignant(CNA+),Mast,Melanocytes(CNA-),MoDC,Mono_CD14,Mono_CD14CD16,Mono_CD16,Myofibroblasts,NK_CD56hiCD16lo,NK_CD56loCD16hi,Neutrophils,PC-cycling,PC-early_RGS13,PC-trans,PC_IGHA,PC_IGHG,Pericytes,SMC,cDC1,cDC2-ISG,cDC2_CD1C,cDC2_CXCL9,cDC2_IL1B,gdT,iCAF_IL6,iCAF_MMP1,mregDC,pDC
0,HAUS4,COL4A3,LY9,FCRL4,CNOT4,CD19,LINC01781,IGHD,NOC2L,COL11A1,PRG4,DIO3OS,HERC5,MAL,IL7R,CD40LG,IGFL2,PARK7,CCR8,ANK3,KIR2DL1,CMPK2,FXYD2,GZMK,GZMK,FGFBP2,FAM166B,PMCH,IL7R,DRAIC,ZNF683,HSPA6,CD1A,GJA5,HSD17B13,STAB2,ESM1,ACKR1,STAC2,DNAJB2,SERPINA9,ACY3,FCER2,SLC4A10,CXCL11,ADAMDEC1,NDUFB2,NLRP3,CXCL5,LYVE1,PLIN2,MUL1,CHIT1,CALML5,CPA3,TYRP1,CLEC10A,S100A12,GPBAR1,LILRA1,CST2,ADGRG3,MYOM2,CTSS,TERT,NCOR1,LINC02576,IGHA2,IGHGP,MUC3A,RERGL,CLEC9A,CLEC4F,FCER1A,CD1B,NAV1,TRDV2,WNT5A,RARRES1,CCL22,CLEC4C
1,LUC7L,LINC01679,CXCR5,TNFRSF13B,STAMBPL1,FCRLA,TNFRSF13B,FCER2,LUC7L,COL10A1,PCOLCE2,IGF1,MX1,TCF7,KLRB1,CD6,IL21,LUC7L,FOXP3,TNF,KLRC3,RSAD2,NELL2,GZMH,KLRG1,S1PR5,KRT86,VCAM1,CD8A,CPA5,GPR25,HSPA1B,IL22RA2,SEMA3G,CD300LG,CCL21,APLN,SELP,TAT,LHFPL2,HTR3A,ELL3,MIR155HG,TRAV1-2,APOBEC3A,C1QA,LUC7L,EREG,TNFSF15,GFRA2,SLAMF9,LUC7L,TREM2,S100P,MS4A2,BANCR,FCER1A,FCN1,LILRA2,FFAR2,RGS4,TRDC,KLRF1,RTN4,TXNDC5,LUC7L,TNFRSF17,IGHA1,IGHG2,KCNJ8,CASQ2,XCR1,P2RY12,CD1C,CD1E,LUC7L,TRDV1,TFPI2,PDPN,AOC1,LILRA4
2,ST13,COCH,CR2,FCRL5,HACD4,MS4A1,BANK1,COL19A1,NDUFB10,ST6GAL2,WNT10B,CXCL14,MX2,CCR7,CCR6,CD3D,NMB,NUMB,GNG8,CDC14A,KIR2DL3,OASL,IL7R,GZMA,DTHD1,FCRL6,LINC02195,TIMD4,PARP8,CRTAM,LINC02446,HSPA1A,CD207,TMEM178A,CA4,PROX1,LY6H,CADM3-AS1,PHGR1,SUMO1,AICDA,MEF2B,STAG3,IL23R,IL27,C1QB,MTMR14,ADGRE3,IL1A,LILRB5,CCL7,MPST,ALOX15B,AARD,TPSAB1,TYR,CD1C,S100A8,FCN1,LILRA5,COL5A3,KLRC1,FGFBP2,PROSER3,BHLHA15,MT-ND2,POU2AF1,JCHAIN,IGHG1,CCDC102B,MYH11,LGALS2,ELAVL4,CLEC10A,CLEC10A,MRPS27,TRGV9,ANGPTL4,FMO1,KCNN1,PTCRA
3,SSU72,AIM2,LINC01857,FCRL2,HACD3,BANK1,BLK,LINC00926,NDUFB11,PLPP4,FAM180B,ABCA10,OAS1,LINC00402,CD40LG,IL32,GNG4,NUP107,LINC02099,TNFAIP3,TRDV1,USP18,CD8B,EOMES,CRTAM,GZMH,IFITM10,CD200R1,CD8B,GNG4,CD8B,DNAJB1,FCGBP,DHH,KIF25,ART4,CHST1,RAB3C,CLXN,HLA-DMB,ACY3,RGS13,CD72,TMIGD2,CXCL10,C1QC,MTMR3,CCL20,CCL3L1,FOLR2,MARCO,MPV17,SLC1A3,MUC5B,TPSB2,FSTL5,FCN1,NFE2,APOBEC3A,GPBAR1,SCG5,XCL2,S1PR5,ATP6V1B2,SHCBP1,MT-ND3,MZB1,DERL3,IGHG3,ABCC9,MYOCD,BATF3,CYP2S1,CD1E,CYP2S1,MRPS28,KIR3DL2,COL7A1,RIPOR3,TREML1,SHD
4,SPSB3,ZNF296,RUBCNL,MS4A1,H1-3,PAX5,COL4A3,FCRL1,NOLC1,LRRC15,HAS1,OSR1,OAS3,TRABD2A,AQP3,CD2,CXCL13,PABPC4,IL2RA,CCNH,KIR3DL2,HERC5,FXYD7,CD8A,TNFSF9,MYBL1,MYO7A,HAVCR2,SCML4,XCL2,CD8A,IFNG,PLEK2,PLLP,TIMP4,SCN3B,PCDH12,SELE,AK5,HHEX,MEF2B,GCSAM,MS4A1,KLRB1,CCL8,ADORA3,NDUFAF3,CCL3L3,MIR3945HG,CD209,SPP1,MTF2,OTOA,MAL2,TPSD1,MLANA,CACNA2D3,S100A9,LILRA5,ICAM4,NTM,SH2D1B,CX3CR1,SLC66A2,IGHV6-1,NCBP2,TXNDC5,TNFRSF17,IGHG4,STEAP4,PLN,FLT3,ANKRD22,LGALS2,CD1C,NAA60,TRDC,LIF,HS3ST3A1,LAMP3,CUX2
5,CLK4,CD83,CD79A,DHRS9,SLC66A3,CR2,MS4A1,LINC02397,NOL9,HSD17B6,MMP27,MFAP4,CMPK2,LINC02273,PTGER4,CD3E,BHLHE40-AS1,PAF1,RTKN2,DDX3Y,KLRC2,MX1,TCF7,CD8B,EOMES,KLRG1,LRRN3,TNFSF4,CCL5,SH2D1A,RGS9,HSPH1,CD1E,VEGFC,BTNL9,RELN,GABRD,SYT15,PPP1R1B,MON1B,ELL3,RAD51,IGHD,COLQ,ANKRD22,TREM2,NDRG3,PTGS2,INHBA,F13A1,SLC11A1,MTFR1,LIPA,CLDN3,HDC,POU3F3,FCGR2B,CFP,LILRA1,PELATON,F2RL2,XCL1,KIR3DL1,DCAF6,H2BC11,NCBP2AS2,IGLL5,IGHV3-74,JSRP1,FAM162B,SGCA,RAB7B,CEACAM4,CFP,IDO1,NAAA,TRGC1,PAPPA,PRRX2,SLCO5A1,KCNK10
6,CLTC,FCRLA,CD83,TLR10,NEDD9,STAP1,TLR10,TCL1A,NOL7,SYNDIG1,AADAC,OGN,IFI44L,PASK,TNFAIP3,CD52,LINC01281,PAFAH1B1,F5,CD69,XCL1,SA

In [ ]:
adata.uns['cosg']['names'].to_csv('data/cosg_genes.csv')